# Gold de perdas — V-Energy

Publica a fila de inspeção e o histórico de inspeções no formato do contrato do portal (`contracts/tabelas/`), a partir das Silvers de leituras, cadastro, medidores, inspeções e faturas e da Gold `gd_uc`. A transformação inteira está neste notebook (sem `%run` e sem módulo `.py` externo). Produz as duas tabelas do contrato:

1. `inspecoes_historicas`: uma inspeção de campo por linha, com a data, o inspetor, o resultado, o tipo de irregularidade e a energia recuperada.
2. `fila_inspecao`: uma UC candidata por linha, com o score (de 0 a 100; quanto maior, mais promissora), a lista de motivos da suspeita e a energia estimada a recuperar em kWh e em R$.

Regras, calibração e dicionário de dados: [`docs/gold/perdas.md`](../../docs/gold/perdas.md).

O padrão `modo=validar` não grava nada: **Run all** valida as fontes, monta e confere as duas tabelas em memória. Para gravar em `v-energy.gold`, troque o widget para `modo=publicar` antes de executar. Quando o Workflow do Databricks existir (#97), a tarefa vai passar `modo=publicar`, como nas outras Golds.

In [0]:
"""Transformações e validações da Gold de perdas."""

from pyspark.sql import DataFrame, Window
from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType

PADRAO_ID_UC = r'^UC\d{6}\z'
PADRAO_ID_INSPECAO = r'^IN\d{7}\z'
PADRAO_INSPETOR = r'^INSP-(00[1-9]|0[1-3][0-9]|040)\z'
CONFIRMADA = 'IRREGULARIDADE CONFIRMADA'
SEM_IRREGULARIDADE = 'SEM IRREGULARIDADE'
RESULTADOS = [CONFIRMADA, SEM_IRREGULARIDADE]
# Tipos da planilha de inspeções (a coluna não tem enum no contrato: fora deles, só aviso)
TIPOS_IRREGULARIDADE = [
    'BY-PASS',
    'DESVIO DIRETO',
    'LACRE VIOLADO',
    'LIGACAO CLANDESTINA',
    'MEDIDOR ADULTERADO',
]
MEDIDOR_SUSPEITO = 'ELETROMECANICO'
DECIMAL_CONTRATO = DecimalType(18, 2)

# Período de comparação do consumo de cada UC: o P0 (jul/2022 a jun/2023), primeiro ano da
# base, anterior a todas as inspeções. O consumo recente é o dos 3 meses antes do mês de
# referência (o seguinte à última leitura, para a fila; o da inspeção, para calibrar).
INICIO_COMPARACAO, FIM_COMPARACAO = '2022-07', '2023-06'
MESES_RECENTES = 3
MESES_ESTIMADAS = 12
MIN_MESES_RECENTES = 2
MIN_MESES_COMPARACAO = 6
# Calibração nas 1.389 inspeções (docs/gold/perdas.md, seção 3): abaixo de 40% de queda o
# acerto fica em torno de 8%; de 40% a 50%, cerca de 56%; acima de 50%, cerca de 93%.
QUEDA_MINIMA = 0.40
QUEDA_DEGRAU = 0.50
PCT_ESTIMADAS_MINIMA = 0.30
PONTOS = {
    'queda_40_50': 35,
    'queda_acima_50': 60,
    'medidor_eletromecanico': 15,
    'leituras_estimadas': 15,
    'desempate_por_queda': 10,
}
SCORE_MAXIMO = sum(PONTOS.values()) - PONTOS['queda_40_50']
# Fator de energia (kWh recuperado ÷ queda mensal) calculado nas inspeções confirmadas;
# fora desta faixa, a calibração está errada ou a fonte mudou.
FATOR_ENERGIA_FAIXA = (1.0, 12.0)
MIN_CONFIRMADAS_CALIBRACAO = 100
# Tarifa média (R$/kWh) plausível para a fila; fora dela, valor ou consumo da fatura estão errados
TARIFA_FAIXA = (0.10, 2.00)
# Colunas, tipos e ordem de contracts/tabelas/<tabela>.yaml. `motivos` é jsonb no contrato:
# a Gold guarda a lista como array<string>, e a exportação (#96) a escreve em JSON.
COLUNAS_CONTRATO = {
    'inspecoes_historicas': [
        ('id_inspecao', 'string'),
        ('id_uc', 'string'),
        ('data', 'date'),
        ('inspetor', 'string'),
        ('resultado', 'string'),
        ('tipo_irregularidade', 'string'),
        ('kwh_recuperado', 'decimal(18,2)'),
    ],
    'fila_inspecao': [
        ('id_uc', 'string'),
        ('score', 'decimal(18,2)'),
        ('motivos', 'array<string>'),
        ('energia_recuperar_kwh', 'decimal(18,2)'),
        ('energia_recuperar_reais', 'decimal(18,2)'),
    ],
}
COLUNAS_FONTES = {
    'leituras': [
        'id_uc',
        'mes_referencia',
        'consumo_kwh',
        'tipo_leitura',
        'flag_troca_medidor',
        'flag_sem_leitura_anterior',
        'flag_sinal_nao_confirmado',
    ],
    'cadastro': ['id_uc', 'subgrupo_tarifario'],
    'medidores': ['id_uc', 'tipo', 'data_instalacao', 'data_retirada'],
    'inspecoes': [c for c, _ in COLUNAS_CONTRATO['inspecoes_historicas']],
    'faturas': ['id_uc', 'mes_referencia', 'valor_energia'],
    'gd_uc': ['id_uc'],
}


def contar(df: DataFrame, regras: dict) -> dict:
    """Conta, numa única passada, as linhas que violam cada regra."""
    linha = df.agg(*[F.count(F.when(cond, 1)).alias(nome) for nome, cond in regras.items()])
    return linha.first().asDict()


def checar(violacoes: dict, rotulo: str, avisos: dict | None = None) -> None:
    """Imprime cada regra e falha se alguma bloqueante tiver violação; avisos só aparecem."""
    for regra, n in violacoes.items():
        print(f'[{"FALHA" if n else "OK   "}] ({rotulo}) {regra}: {n:,}')
    for regra, n in (avisos or {}).items():
        print(f'[{"AVISO" if n else "OK   "}] ({rotulo}) {regra}: {n:,}')
    falhas = {regra: n for regra, n in violacoes.items() if n}
    if falhas:
        raise RuntimeError(f'Validações bloqueantes falharam ({rotulo}): {falhas}')


def fora_do_padrao(coluna: str, padrao: str):
    return F.col(coluna).isNull() | ~F.col(coluna).rlike(padrao)


def vazio(coluna: str):
    return F.coalesce(F.trim(F.col(coluna)), F.lit('')) == ''


def mes(coluna_aaaa_mm):
    """Primeiro dia do mês de um texto AAAA-MM."""
    return F.to_date(F.concat(coluna_aaaa_mm, F.lit('-01')))


def validar_fontes(fontes: dict) -> None:
    """Colunas, chaves e domínios das fontes, antes de qualquer transformação."""
    for nome, colunas in COLUNAS_FONTES.items():
        ausentes = sorted(set(colunas) - set(fontes[nome].columns))
        if ausentes:
            raise RuntimeError(f'Fonte {nome} sem as colunas {ausentes}')
    inspecoes, cadastro = fontes['inspecoes'], fontes['cadastro']
    violacoes = contar(
        inspecoes,
        {
            'inspeções: id_inspecao fora do padrão IN + 7 dígitos': fora_do_padrao(
                'id_inspecao', PADRAO_ID_INSPECAO
            ),
            'inspeções: id_uc fora do padrão UC + 6 dígitos': fora_do_padrao('id_uc', PADRAO_ID_UC),
            'inspeções: data nula': F.col('data').isNull(),
            'inspeções: resultado fora do domínio': ~F.coalesce(
                F.col('resultado').isin(RESULTADOS), F.lit(False)
            ),
        },
    )
    violacoes |= contar(
        fontes['leituras'],
        {
            'leituras: id_uc fora do padrão': fora_do_padrao('id_uc', PADRAO_ID_UC),
            'leituras: mês de referência fora de AAAA-MM': fora_do_padrao(
                'mes_referencia', r'^\d{4}-(0[1-9]|1[0-2])\z'
            ),
            'leituras: tipo fora de REAL / ESTIMADA': ~F.coalesce(
                F.col('tipo_leitura').isin('REAL', 'ESTIMADA'), F.lit(False)
            ),
        },
    )
    violacoes |= {
        'inspeções: id_inspecao repetido (linhas a mais)': inspecoes.count()
        - inspecoes.select('id_inspecao').distinct().count(),
        'leituras: id_uc × mês repetido (linhas a mais)': fontes['leituras'].count()
        - fontes['leituras'].select('id_uc', 'mes_referencia').distinct().count(),
        'cadastro: id_uc repetido (linhas a mais)': cadastro.count()
        - cadastro.select('id_uc').distinct().count(),
        'inspeções: id_uc ausente da Silver de cadastro': inspecoes.join(
            cadastro.select('id_uc'), 'id_uc', 'left_anti'
        ).count(),
    }
    checar(violacoes, 'fontes')


def leituras_mensais(leituras: DataFrame) -> DataFrame:
    """Uma linha por UC e mês, com o consumo utilizável e a marca de leitura estimada.

    Utilizável segue docs/silver/silver_leituras.md (seção 5): consumo ≥ 0 e sem as flags
    de troca de medidor, de leitura anterior ausente e de sinal não confirmado.
    """
    impeditiva = (
        F.col('flag_troca_medidor')
        | F.col('flag_sem_leitura_anterior')
        | F.col('flag_sinal_nao_confirmado')
    )
    return leituras.select(
        'id_uc',
        'mes_referencia',
        mes(F.col('mes_referencia')).alias('mes'),
        F.when((F.col('consumo_kwh') >= 0) & ~impeditiva, F.col('consumo_kwh')).alias('kwh'),
        (F.col('tipo_leitura') == 'ESTIMADA').cast('int').alias('estimada'),
    )


def sinais(mensal: DataFrame, referencias: DataFrame, medidores: DataFrame) -> DataFrame:
    """Sinais de cada referência (chave, id_uc, mes_ref, data_ref), só com meses anteriores.

    Para a fila, a referência é o mês seguinte à última leitura; para calibrar, é o mês da
    inspeção. O medidor é o instalado na data de referência.
    """
    inicio_ref, fim_ref = mes(F.lit(INICIO_COMPARACAO)), mes(F.lit(FIM_COMPARACAO))
    recente = (F.col('mes') >= F.add_months('mes_ref', -MESES_RECENTES)) & (
        F.col('mes') < F.col('mes_ref')
    )
    no_p0 = (F.col('mes') >= inicio_ref) & (F.col('mes') <= fim_ref)
    ultimo_ano = (F.col('mes') >= F.add_months('mes_ref', -MESES_ESTIMADAS)) & (
        F.col('mes') < F.col('mes_ref')
    )
    agregado = (
        referencias.join(mensal, 'id_uc')
        .groupBy('chave', 'id_uc', 'mes_ref', 'data_ref')
        .agg(
            F.avg(F.when(recente, F.col('kwh'))).alias('consumo_recente'),
            F.count(F.when(recente, F.col('kwh'))).alias('meses_recentes'),
            F.avg(F.when(no_p0, F.col('kwh'))).alias('consumo_comparacao'),
            F.count(F.when(no_p0, F.col('kwh'))).alias('meses_comparacao'),
            F.avg(F.when(ultimo_ano, F.col('estimada'))).alias('pct_estimadas'),
        )
    )
    no_dia = medidores.alias('m')
    medidor = (
        referencias.alias('r')
        .join(
            no_dia,
            (F.col('m.id_uc') == F.col('r.id_uc'))
            & (F.col('m.data_instalacao') <= F.col('r.data_ref'))
            & (
                F.col('m.data_retirada').isNull() | (F.col('m.data_retirada') > F.col('r.data_ref'))
            ),
        )
        .groupBy(F.col('r.chave').alias('chave'))
        .agg(
            F.max_by(F.col('m.tipo'), F.col('m.data_instalacao')).alias('tipo_medidor'),
            F.count(F.lit(1)).alias('medidores_na_data'),
        )
    )
    suficiente = (
        (F.col('meses_recentes') >= MIN_MESES_RECENTES)
        & (F.col('meses_comparacao') >= MIN_MESES_COMPARACAO)
        & (F.col('consumo_comparacao') > 0)
    )
    return (
        agregado.join(medidor, 'chave', 'left')
        .withColumn(
            'queda',
            F.when(suficiente, 1 - F.col('consumo_recente') / F.col('consumo_comparacao')),
        )
        .withColumn(
            'queda_kwh',
            F.when(suficiente, F.col('consumo_comparacao') - F.col('consumo_recente')),
        )
    )


def pontuar(df: DataFrame) -> DataFrame:
    """Score aditivo (0 a 100) para quem tem queda ≥ QUEDA_MINIMA; nulo para os demais."""
    pontos = (
        F.when(F.col('queda') >= QUEDA_DEGRAU, PONTOS['queda_acima_50']).otherwise(
            PONTOS['queda_40_50']
        )
        + F.when(
            F.col('tipo_medidor') == MEDIDOR_SUSPEITO, PONTOS['medidor_eletromecanico']
        ).otherwise(0)
        + F.when(
            F.col('pct_estimadas') >= PCT_ESTIMADAS_MINIMA, PONTOS['leituras_estimadas']
        ).otherwise(0)
        + PONTOS['desempate_por_queda'] * F.col('queda')
    )
    return df.withColumn('score', F.when(F.col('queda') >= QUEDA_MINIMA, F.round(pontos, 2)))


def referencias_das_inspecoes(inspecoes: DataFrame) -> DataFrame:
    return inspecoes.select(
        F.col('id_inspecao').alias('chave'),
        'id_uc',
        F.trunc('data', 'MM').alias('mes_ref'),
        F.col('data').alias('data_ref'),
    )


def calibrar(sinais_inspecoes: DataFrame, inspecoes: DataFrame, gd_uc: DataFrame) -> dict:
    """Fator de energia e ordenação do score medidos nas inspeções de UCs sem GD.

    Usa só dados anteriores a cada inspeção, como a fila usa só dados anteriores à referência.
    """
    base = sinais_inspecoes.join(
        inspecoes.select(F.col('id_inspecao').alias('chave'), 'resultado', 'kwh_recuperado'),
        'chave',
    ).join(gd_uc.select('id_uc'), 'id_uc', 'left_anti')
    confirmadas = base.filter((F.col('resultado') == CONFIRMADA) & (F.col('queda') >= QUEDA_MINIMA))
    fator = confirmadas.agg(
        F.count(F.lit(1)).alias('n'),
        F.expr('percentile(kwh_recuperado / queda_kwh, 0.5)').alias('fator'),
        F.corr(F.col('kwh_recuperado').cast('double'), 'queda_kwh').alias('correlacao'),
    ).first()
    acerto = F.avg((F.col('resultado') == CONFIRMADA).cast('int'))
    ordem = (
        pontuar(base)
        .filter(F.col('score').isNotNull())
        .agg(
            acerto.alias('geral'),
            F.avg(
                F.when(F.col('score') >= 60, (F.col('resultado') == CONFIRMADA).cast('int'))
            ).alias('alto'),
            F.avg(
                F.when(F.col('score') < 60, (F.col('resultado') == CONFIRMADA).cast('int'))
            ).alias('baixo'),
        )
        .first()
    )
    return {
        'confirmadas': fator['n'],
        'fator_energia': fator['fator'],
        'correlacao': fator['correlacao'],
        'acerto_score_60_ou_mais': ordem['alto'],
        'acerto_score_abaixo_60': ordem['baixo'],
    }


def validar_calibracao(calibracao: dict) -> None:
    fator = calibracao['fator_energia']
    alto, baixo = calibracao['acerto_score_60_ou_mais'], calibracao['acerto_score_abaixo_60']
    checar(
        {
            f'calibração: menos de {MIN_CONFIRMADAS_CALIBRACAO} confirmadas com queda ≥ '
            f'{QUEDA_MINIMA:.0%}': int(calibracao['confirmadas'] < MIN_CONFIRMADAS_CALIBRACAO),
            f'calibração: fator de energia fora de {FATOR_ENERGIA_FAIXA}': int(
                fator is None or not FATOR_ENERGIA_FAIXA[0] <= fator <= FATOR_ENERGIA_FAIXA[1]
            ),
            'calibração: score ≥ 60 não acerta mais que score < 60 (score não ordena)': int(
                alto is None or baixo is None or alto <= baixo
            ),
        },
        'calibração',
    )
    print(
        f'Fator de energia: {fator:.4f} kWh recuperado por kWh de queda mensal '
        f'({calibracao["confirmadas"]:,} confirmadas; correlação {calibracao["correlacao"]:.3f})'
    )
    print(f'Acerto nas inspeções: score ≥ 60 = {alto:.1%} | score < 60 = {baixo:.1%}')


def tarifas(faturas: DataFrame, mensal: DataFrame, cadastro: DataFrame, mes_ref) -> DataFrame:
    """Tarifa média (R$/kWh) de cada UC nos 12 meses antes da referência.

    Valor da energia ÷ consumo utilizável nos meses com fatura e leitura; sem esses meses,
    a mediana do subgrupo tarifário.
    """
    ultimo_ano = (F.col('mes') >= F.add_months(F.lit(mes_ref), -MESES_ESTIMADAS)) & (
        F.col('mes') < F.lit(mes_ref)
    )
    por_uc = (
        faturas.select('id_uc', 'mes_referencia', 'valor_energia')
        .join(mensal.filter(ultimo_ano & (F.col('kwh') > 0)), ['id_uc', 'mes_referencia'])
        .groupBy('id_uc')
        .agg((F.sum('valor_energia') / F.sum('kwh')).cast('double').alias('tarifa_uc'))
    )
    com_subgrupo = cadastro.select('id_uc', 'subgrupo_tarifario').join(por_uc, 'id_uc', 'left')
    mediana = com_subgrupo.groupBy('subgrupo_tarifario').agg(
        F.expr('percentile(tarifa_uc, 0.5)').alias('tarifa_subgrupo')
    )
    return com_subgrupo.join(mediana, 'subgrupo_tarifario', 'left').select(
        'id_uc',
        F.coalesce('tarifa_uc', 'tarifa_subgrupo').alias('tarifa'),
        F.when(F.col('tarifa_uc').isNotNull(), 'UC').otherwise('SUBGRUPO').alias('origem_tarifa'),
    )


def ultima_inspecao(inspecoes: DataFrame) -> DataFrame:
    ordem = Window.partitionBy('id_uc').orderBy(F.desc('data'), F.desc('id_inspecao'))
    return (
        inspecoes.withColumn('ordem', F.row_number().over(ordem))
        .filter(F.col('ordem') == 1)
        .select(
            'id_uc',
            F.col('data').alias('data_ultima_inspecao'),
            F.col('resultado').alias('resultado_ultima_inspecao'),
        )
    )


def construir_inspecoes_historicas(inspecoes: DataFrame) -> DataFrame:
    return inspecoes.select(
        'id_inspecao',
        'id_uc',
        F.col('data').cast('date').alias('data'),
        'inspetor',
        'resultado',
        'tipo_irregularidade',
        F.col('kwh_recuperado').cast(DECIMAL_CONTRATO).alias('kwh_recuperado'),
    )


def construir_candidatas(fontes: dict, mensal: DataFrame, mes_ref, fator: float) -> DataFrame:
    """Todas as UCs do cadastro com sinais, score, exclusões e energia (base da fila)."""
    referencias = fontes['cadastro'].select(
        F.col('id_uc').alias('chave'),
        'id_uc',
        F.lit(mes_ref).alias('mes_ref'),
        F.lit(mes_ref).alias('data_ref'),
    )
    gd = fontes['gd_uc'].select('id_uc').distinct().withColumn('com_gd', F.lit(True))
    return (
        pontuar(sinais(mensal, referencias, fontes['medidores']))
        .join(gd, 'id_uc', 'left')
        .join(ultima_inspecao(fontes['inspecoes']), 'id_uc', 'left')
        .join(tarifas(fontes['faturas'], mensal, fontes['cadastro'], mes_ref), 'id_uc', 'left')
        .withColumn('com_gd', F.coalesce('com_gd', F.lit(False)))
        .withColumn(
            'situacao',
            F.when(F.col('score').isNull(), 'SEM QUEDA SUFICIENTE')
            .when(F.col('com_gd'), 'FORA: GD (R4)')
            .when(F.col('resultado_ultima_inspecao') == CONFIRMADA, 'FORA: FRAUDE JÁ CONFIRMADA')
            .when(F.col('resultado_ultima_inspecao') == SEM_IRREGULARIDADE, 'FILA: REINSPEÇÃO')
            .otherwise('FILA: NUNCA INSPECIONADA'),
        )
        .withColumn('energia_kwh', F.round(F.lit(fator) * F.col('queda_kwh'), 2))
        .withColumn('energia_reais', F.round(F.col('energia_kwh') * F.col('tarifa'), 2))
    )


def motivos(mes_ref) -> F.Column:
    """Lista de textos, cada sinal com o seu valor: o formato do importador e do backend."""
    rotulo = 'MM/yyyy'
    ini_recente = F.date_format(F.add_months(F.lit(mes_ref), -MESES_RECENTES), rotulo)
    ini_ano = F.date_format(F.add_months(F.lit(mes_ref), -MESES_ESTIMADAS), rotulo)
    fim = F.date_format(F.add_months(F.lit(mes_ref), -1), rotulo)
    ini_p0 = F.date_format(mes(F.lit(INICIO_COMPARACAO)), rotulo)
    fim_p0 = F.date_format(mes(F.lit(FIM_COMPARACAO)), rotulo)
    lista = F.array(
        F.format_string(
            'Consumo %d%% abaixo do primeiro ano da própria UC: %d kWh/mês de %s a %s, '
            'contra %d kWh/mês de %s a %s',
            F.round(100 * F.col('queda')).cast('int'),
            F.round('consumo_recente').cast('int'),
            ini_recente,
            fim,
            F.round('consumo_comparacao').cast('int'),
            ini_p0,
            fim_p0,
        ),
        F.when(
            F.col('pct_estimadas') >= PCT_ESTIMADAS_MINIMA,
            F.format_string(
                '%d%% das leituras estimadas de %s a %s',
                F.round(100 * F.col('pct_estimadas')).cast('int'),
                ini_ano,
                fim,
            ),
        ),
        F.when(F.col('tipo_medidor') == MEDIDOR_SUSPEITO, F.lit('Medidor eletromecânico')),
        F.when(
            F.col('resultado_ultima_inspecao') == SEM_IRREGULARIDADE,
            F.format_string(
                'Inspecionada em %s sem irregularidade constatada',
                F.date_format('data_ultima_inspecao', 'dd/MM/yyyy'),
            ),
        ),
    )
    return F.filter(lista, lambda m: m.isNotNull())


def construir_fila(candidatas: DataFrame, mes_ref) -> DataFrame:
    return candidatas.filter(F.col('situacao').startswith('FILA')).select(
        'id_uc',
        F.col('score').cast(DECIMAL_CONTRATO).alias('score'),
        motivos(mes_ref).alias('motivos'),
        F.col('energia_kwh').cast(DECIMAL_CONTRATO).alias('energia_recuperar_kwh'),
        F.col('energia_reais').cast(DECIMAL_CONTRATO).alias('energia_recuperar_reais'),
    )


def validar_gold(tabelas: dict, fontes: dict, rotulo: str) -> None:
    """Contrato, chaves, R4, exclusões e reconciliação das duas tabelas."""
    historico, fila = tabelas['inspecoes_historicas'], tabelas['fila_inspecao']
    for nome, df in tabelas.items():
        colunas = [(c.name, c.dataType.simpleString()) for c in df.schema]
        if colunas != COLUNAS_CONTRATO[nome]:
            raise RuntimeError(f'{nome}: colunas, tipos ou ordem fora do contrato: {colunas}')

    obrigatorias = [
        c for c, _ in COLUNAS_CONTRATO['inspecoes_historicas'] if c != 'tipo_irregularidade'
    ]
    violacoes = contar(
        historico,
        {
            'inspecoes_historicas: coluna obrigatória nula': F.greatest(
                *[F.col(c).isNull() for c in obrigatorias]
            ),
            'inspecoes_historicas: id_inspecao fora do padrão': fora_do_padrao(
                'id_inspecao', PADRAO_ID_INSPECAO
            ),
            'inspecoes_historicas: id_uc fora do padrão': fora_do_padrao('id_uc', PADRAO_ID_UC),
            'inspecoes_historicas: inspetor fora do padrão INSP-001 a INSP-040': fora_do_padrao(
                'inspetor', PADRAO_INSPETOR
            ),
            'inspecoes_historicas: resultado fora do domínio': ~F.col('resultado').isin(RESULTADOS),
            'inspecoes_historicas: kwh_recuperado negativo': F.col('kwh_recuperado') < 0,
            'inspecoes_historicas: confirmada sem tipo de irregularidade': (
                F.col('resultado') == CONFIRMADA
            )
            & vazio('tipo_irregularidade'),
            'inspecoes_historicas: sem irregularidade com tipo ou energia recuperada': (
                F.col('resultado') == SEM_IRREGULARIDADE
            )
            & (F.col('tipo_irregularidade').isNotNull() | (F.col('kwh_recuperado') != 0)),
        },
    )
    avisos = contar(
        historico,
        {
            f'inspecoes_historicas: tipo fora dos {len(TIPOS_IRREGULARIDADE)} da planilha': F.col(
                'tipo_irregularidade'
            ).isNotNull()
            & ~F.col('tipo_irregularidade').isin(TIPOS_IRREGULARIDADE),
        },
    )
    violacoes |= contar(
        fila,
        {
            'fila_inspecao: coluna nula': F.greatest(*[F.col(c).isNull() for c in fila.columns]),
            'fila_inspecao: id_uc fora do padrão': fora_do_padrao('id_uc', PADRAO_ID_UC),
            f'fila_inspecao: score fora de 0 a {SCORE_MAXIMO}': (F.col('score') < 0)
            | (F.col('score') > SCORE_MAXIMO),
            'fila_inspecao: motivos vazio ou com texto vazio': (F.size('motivos') < 1)
            | F.exists('motivos', lambda m: m.isNull() | (F.trim(m) == '')),
            'fila_inspecao: energia a recuperar negativa': (F.col('energia_recuperar_kwh') < 0)
            | (F.col('energia_recuperar_reais') < 0),
            f'fila_inspecao: tarifa média implícita (R$ ÷ kWh) fora de {TARIFA_FAIXA}': (
                F.col('energia_recuperar_kwh') > 0
            )
            & ~(F.col('energia_recuperar_reais') / F.col('energia_recuperar_kwh')).between(
                *TARIFA_FAIXA
            ),
        },
    )

    silver = (
        fontes['inspecoes']
        .agg(F.count(F.lit(1)).alias('linhas'), F.sum('kwh_recuperado').alias('kwh'))
        .first()
    )
    total = historico.agg(
        F.count(F.lit(1)).alias('linhas'),
        F.countDistinct('id_inspecao').alias('ids'),
        F.sum('kwh_recuperado').alias('kwh'),
    ).first()
    total_fila = fila.agg(
        F.count(F.lit(1)).alias('linhas'), F.countDistinct('id_uc').alias('ids')
    ).first()
    confirmadas = ultima_inspecao(fontes['inspecoes']).filter(
        F.col('resultado_ultima_inspecao') == CONFIRMADA
    )
    violacoes |= {
        'inspecoes_historicas: linhas diferentes da Silver': abs(
            total['linhas'] - silver['linhas']
        ),
        'inspecoes_historicas: id_inspecao repetido (linhas a mais)': total['linhas']
        - total['ids'],
        'inspecoes_historicas: kWh recuperado total diferente da Silver': int(
            total['kwh'] != silver['kwh']
        ),
        'inspecoes_historicas: id_uc ausente do cadastro (dim_uc)': historico.join(
            fontes['cadastro'].select('id_uc'), 'id_uc', 'left_anti'
        ).count(),
        'fila_inspecao: id_uc repetido (linhas a mais)': total_fila['linhas'] - total_fila['ids'],
        'fila_inspecao: id_uc ausente do cadastro (dim_uc)': fila.join(
            fontes['cadastro'].select('id_uc'), 'id_uc', 'left_anti'
        ).count(),
        'fila_inspecao: UC com GD na fila (R4)': fila.join(
            fontes['gd_uc'].select('id_uc'), 'id_uc', 'left_semi'
        ).count(),
        'fila_inspecao: UC cuja última inspeção confirmou fraude': fila.join(
            confirmadas, 'id_uc', 'left_semi'
        ).count(),
    }
    checar(violacoes, rotulo, avisos)
    print(
        f'Reconciliação ({rotulo}): {silver["linhas"]:,} inspeções | '
        f'{silver["kwh"]:,} kWh recuperados | {total_fila["linhas"]:,} UCs na fila'
    )


def publicar_delta(spark, tabelas: dict, destinos: dict, validar) -> tuple:
    """Publica e confere as tabelas Gold; recupera os destinos afetados em caso de falha."""
    anteriores = {}
    for nome in tabelas:
        destino = destinos[nome]
        if spark.catalog.tableExists(destino):
            detalhe = spark.sql(f'DESCRIBE DETAIL {destino}').first()
            if detalhe['format'] != 'delta':
                raise RuntimeError(f'O destino precisa ser Delta: {destino}')
            anteriores[nome] = spark.sql(f'DESCRIBE HISTORY {destino} LIMIT 1').first()['version']
        else:
            anteriores[nome] = None

    afetadas = []
    persistidas, versoes = {}, {}
    try:
        for nome, df in tabelas.items():
            destino = destinos[nome]
            afetadas.append(nome)
            (
                df.write.format('delta')
                .mode('overwrite')
                .option('overwriteSchema', 'true')
                .saveAsTable(destino)
            )
            versao = spark.sql(f'DESCRIBE HISTORY {destino} LIMIT 1').first()['version']
            persistido = spark.read.option('versionAsOf', versao).table(destino)
            esperado = [(c.name, c.dataType.simpleString()) for c in df.schema]
            gravado = [(c.name, c.dataType.simpleString()) for c in persistido.schema]
            if esperado != gravado:
                raise RuntimeError(f'Schema persistido divergente: {destino}')
            if (
                df.exceptAll(persistido).limit(1).count()
                or persistido.exceptAll(df).limit(1).count()
            ):
                raise RuntimeError(f'Conteúdo persistido divergente: {destino}')
            persistidas[nome] = persistido
            versoes[nome] = {'tabela': destino, 'versao_delta': versao}
        validar(persistidas)
    except Exception as erro:
        falhas = []
        for nome in reversed(afetadas):
            destino = destinos[nome]
            try:
                versao = anteriores[nome]
                if versao is None:
                    spark.sql(f'DROP TABLE IF EXISTS {destino}')
                else:
                    spark.sql(f'RESTORE TABLE {destino} TO VERSION AS OF {versao}')
            except Exception as falha:
                falhas.append(
                    RuntimeError(
                        f'Falha ao recuperar {destino} (versão anterior: {versao}): {falha}'
                    )
                )
        if falhas:
            raise ExceptionGroup(
                'Publicação falhou e a recuperação ficou incompleta', [erro, *falhas]
            ) from erro
        raise
    return persistidas, versoes

In [0]:
# ruff: noqa: F821
import re

parametros = {
    'tabela_leituras': 'v-energy.silver.tb_leituras',
    'tabela_cadastro': 'v-energy.silver.tb_unidades_consumidoras',
    'tabela_medidores': 'v-energy.silver.tb_medidores',
    'tabela_inspecoes': 'v-energy.silver.tb_inspecoes',
    'tabela_faturas_publicacao': 'v-energy.silver.tb_faturas_publicacao',
    'tabela_gd_uc': 'v-energy.gold.gd_uc',
    'schema_gold': 'v-energy.gold',
    'modo': 'validar',
}
for nome, padrao in parametros.items():
    dbutils.widgets.text(nome, padrao)
config = {nome: dbutils.widgets.get(nome).strip() for nome in parametros}
if config['modo'] not in {'validar', 'publicar'}:
    raise RuntimeError('modo deve ser validar ou publicar')


def qualificar(nome, partes=3):
    segmentos = nome.split('.')
    if len(segmentos) != partes or any(
        not re.fullmatch(r'[A-Za-z0-9_-]+', segmento) for segmento in segmentos
    ):
        raise RuntimeError(f'Identificador inválido: {nome}')
    return '.'.join(f'`{segmento}`' for segmento in segmentos)


schema_gold = qualificar(config['schema_gold'], 2)
if config['schema_gold'].split('.')[1] in {'silver', 'bronze', 'default'}:
    raise RuntimeError('O destino deve ser um schema próprio da Gold')
fontes_lidas = {}
print(f'Schema Gold: {schema_gold} | Modo: {config["modo"]}')
if config['modo'] == 'publicar':
    print(f'ATENÇÃO: modo=publicar. As tabelas Gold serão gravadas em {schema_gold}')

Schema Gold: `v-energy`.`gold` | Modo: publicar
ATENÇÃO: modo=publicar. As tabelas Gold serão gravadas em `v-energy`.`gold`


## 1. Leitura consistente das fontes (`VERSION AS OF`) e validação

Fixa a versão Delta atual de cada fonte no início da execução, para que validações, transformações e escrita leiam exatamente os mesmos dados. As faturas são lidas pelo marcador `tb_faturas_publicacao`, na revisão imutável que ele indica (`docs/auditoria_dados/operacao_faturas.md`), e não pelo nome final, que pode estar no meio de uma publicação. Depois, calibra o fator de energia e confere que o score ordena as inspeções passadas, monta as duas tabelas e confere o contrato.

In [0]:
# ruff: noqa: F821
def ler_versao(nome_qualificado, chave, versao=None):
    if not spark.catalog.tableExists(nome_qualificado):
        raise RuntimeError(
            f'Tabela não encontrada: {nome_qualificado}. Publique a fonte ou ajuste o widget.'
        )
    detalhe = spark.sql(f'DESCRIBE DETAIL {nome_qualificado}').first()
    if detalhe['format'] != 'delta':
        raise RuntimeError(f'A fonte precisa ser Delta: {nome_qualificado}')
    if versao is None:
        versao = spark.sql(f'DESCRIBE HISTORY {nome_qualificado} LIMIT 1').first()['version']
    fontes_lidas[chave] = {
        'tabela': nome_qualificado,
        'id_delta': detalhe['id'],
        'versao_delta': versao,
    }
    print(f'{chave}: {nome_qualificado} (versão {versao})')
    return spark.read.option('versionAsOf', versao).table(nome_qualificado)


def ler_faturas_publicadas():
    """Contrato do consumidor de docs/auditoria_dados/operacao_faturas.md."""
    marcador = qualificar(config['tabela_faturas_publicacao'])
    if not spark.catalog.tableExists(marcador):
        raise RuntimeError(f'Publicação da Silver de faturas ausente: {marcador}')
    linhas = spark.table(marcador).collect()
    catalogo, schema, _ = config['tabela_faturas_publicacao'].split('.')
    esperados = {
        qualificar(f'{catalogo}.{schema}.tb_faturas'),
        qualificar(f'{catalogo}.{schema}.tb_faturas_pendentes_cadastro'),
    }
    if (
        len(linhas) != 2
        or {r['destino'] for r in linhas} != esperados
        or len({r['id_execucao'] for r in linhas}) != 1
    ):
        raise RuntimeError('Publicação da Silver de faturas ausente ou incompleta')
    linha = next(r for r in linhas if r['destino'] == qualificar(f'{catalogo}.{schema}.tb_faturas'))
    fonte = qualificar(linha['fonte'].replace('`', ''))
    return ler_versao(fonte, 'faturas', linha['versao_fonte'])


fontes = {
    'leituras': ler_versao(qualificar(config['tabela_leituras']), 'leituras'),
    'cadastro': ler_versao(qualificar(config['tabela_cadastro']), 'cadastro'),
    'medidores': ler_versao(qualificar(config['tabela_medidores']), 'medidores'),
    'inspecoes': ler_versao(qualificar(config['tabela_inspecoes']), 'inspecoes'),
    'faturas': ler_faturas_publicadas(),
    'gd_uc': ler_versao(qualificar(config['tabela_gd_uc']), 'gd_uc'),
}
validar_fontes(fontes)

mensal = leituras_mensais(fontes['leituras'])
ultimo_mes = mensal.agg(F.max('mes')).first()[0]
mes_ref = ultimo_mes.replace(day=1)
mes_ref = mes_ref.replace(year=mes_ref.year + mes_ref.month // 12, month=mes_ref.month % 12 + 1)
if mes_ref.strftime('%Y-%m') <= FIM_COMPARACAO:
    raise RuntimeError(f'A referência ({FIM_COMPARACAO}) precisa ser anterior a {mes_ref:%Y-%m}')
print(
    f'Mês de referência da fila: {mes_ref:%m/%Y} (última leitura em {ultimo_mes:%m/%Y}) | '
    f'período de comparação (P0): {INICIO_COMPARACAO} a {FIM_COMPARACAO}'
)

sinais_inspecoes = sinais(
    mensal, referencias_das_inspecoes(fontes['inspecoes']), fontes['medidores']
)
calibracao = calibrar(sinais_inspecoes, fontes['inspecoes'], fontes['gd_uc'])
validar_calibracao(calibracao)

candidatas = construir_candidatas(fontes, mensal, mes_ref, calibracao['fator_energia'])
checar(
    {
        'fila: UC candidata sem tarifa': candidatas.filter(
            F.col('situacao').startswith('FILA') & F.col('tarifa').isNull()
        ).count(),
        'fila: UC candidata com mais de um medidor na data de referência': candidatas.filter(
            F.col('situacao').startswith('FILA') & (F.col('medidores_na_data') > 1)
        ).count(),
    },
    'candidatas',
)
tabelas = {
    'inspecoes_historicas': construir_inspecoes_historicas(fontes['inspecoes']),
    'fila_inspecao': construir_fila(candidatas, mes_ref),
}
validar_gold(tabelas, fontes, 'antes de gravar')
for nome, df_tab in tabelas.items():
    print(f'[OK] {nome}: {df_tab.count():,} registros validados em memória')

leituras: `v-energy`.`silver`.`tb_leituras` (versão 4)
cadastro: `v-energy`.`silver`.`tb_unidades_consumidoras` (versão 43)
medidores: `v-energy`.`silver`.`tb_medidores` (versão 1)
inspecoes: `v-energy`.`silver`.`tb_inspecoes` (versão 2)
faturas: `v-energy`.`silver_revisoes`.`tb_faturas_revisao_921a5cb56f964cebaafd62041472417c` (versão 0)
gd_uc: `v-energy`.`gold`.`gd_uc` (versão 3)
[OK   ] (fontes) inspeções: id_inspecao fora do padrão IN + 7 dígitos: 0
[OK   ] (fontes) inspeções: id_uc fora do padrão UC + 6 dígitos: 0
[OK   ] (fontes) inspeções: data nula: 0
[OK   ] (fontes) inspeções: resultado fora do domínio: 0
[OK   ] (fontes) leituras: id_uc fora do padrão: 0
[OK   ] (fontes) leituras: mês de referência fora de AAAA-MM: 0
[OK   ] (fontes) leituras: tipo fora de REAL / ESTIMADA: 0
[OK   ] (fontes) inspeções: id_inspecao repetido (linhas a mais): 0
[OK   ] (fontes) leituras: id_uc × mês repetido (linhas a mais): 0
[OK   ] (fontes) cadastro: id_uc repetido (linhas a mais): 0
[OK   ]

## 2. Verificação da pergunta do Gerente de Perdas

**Rafael Coutinho:** _"Quero uma fila das unidades suspeitas, da mais promissora para a menos promissora, com o motivo da suspeita e quanta energia dá para recuperar."_ Confere nas tabelas geradas e nas inspeções passadas:
- o acerto das inspeções por faixa de queda de consumo, medida só com dados anteriores a cada inspeção (a base do corte de 40%);
- o acerto por faixa de score (o score ordena?);
- a concentração das inspeções em poucos bairros (por que bairro e classe não pontuam);
- a composição da fila, os totais de energia e as UCs que ficaram de fora;
- as 10 primeiras UCs da fila, com os motivos;
- a tarifa usada na conversão para R$;
- o consumo antes e depois da inspeção (por que as UCs com fraude confirmada saem da fila);
- a medição antiga do acerto ("92%"), que compara anos inteiros e inclui meses depois da inspeção.

In [0]:
# ruff: noqa: F821
acerto = F.round(100 * F.avg((F.col('resultado') == CONFIRMADA).cast('int')), 1)


def acerto_se(condicao):
    return F.round(100 * F.avg(F.when(condicao, (F.col('resultado') == CONFIRMADA).cast('int'))), 1)


inspecionadas = sinais_inspecoes.join(
    fontes['inspecoes'].select(F.col('id_inspecao').alias('chave'), 'resultado'), 'chave'
).join(fontes['gd_uc'].select('id_uc', F.lit(True).alias('com_gd')), 'id_uc', 'left')

print('=== 1) Acerto das inspeções por queda de consumo antes da inspeção (contra o P0) ===')
display(
    inspecionadas.withColumn(
        'faixa_queda',
        F.when(F.col('com_gd'), '0. UC com GD')
        .when(F.col('queda').isNull(), '0. sem leituras suficientes')
        .when(F.col('queda') < 0.30, '1. menos de 30%')
        .when(F.col('queda') < QUEDA_MINIMA, '2. de 30% a 40%')
        .when(F.col('queda') < QUEDA_DEGRAU, '3. de 40% a 50%')
        .otherwise('4. 50% ou mais'),
    )
    .groupBy('faixa_queda')
    .agg(
        F.count(F.lit(1)).alias('inspecoes'),
        F.sum((F.col('resultado') == CONFIRMADA).cast('int')).alias('confirmadas'),
        acerto.alias('pct_acerto'),
        acerto_se(F.col('tipo_medidor') == MEDIDOR_SUSPEITO).alias('pct_eletromecanico'),
        acerto_se(F.coalesce(F.col('tipo_medidor') != MEDIDOR_SUSPEITO, F.lit(True))).alias(
            'pct_outros_medidores'
        ),
        acerto_se(F.col('pct_estimadas') >= PCT_ESTIMADAS_MINIMA).alias('pct_estimadas_30_ou_mais'),
        acerto_se(F.col('pct_estimadas') < PCT_ESTIMADAS_MINIMA).alias('pct_estimadas_menos_30'),
    )
    .orderBy('faixa_queda')
)

print('=== 2) Acerto das inspeções por faixa de score (UCs sem GD com queda ≥ 40%) ===')
display(
    pontuar(inspecionadas.filter(F.col('com_gd').isNull()))
    .filter(F.col('score').isNotNull())
    .withColumn(
        'faixa_score',
        F.when(F.col('score') >= 90, '1. 90 ou mais')
        .when(F.col('score') >= 75, '2. 75 a 90')
        .when(F.col('score') >= 60, '3. 60 a 75')
        .otherwise('4. menos de 60'),
    )
    .groupBy('faixa_score')
    .agg(
        F.count(F.lit(1)).alias('inspecoes'),
        F.sum((F.col('resultado') == CONFIRMADA).cast('int')).alias('confirmadas'),
        acerto.alias('pct_acerto'),
    )
    .orderBy('faixa_score')
)

print('=== 3) Concentração das inspeções: os 4 bairros mais inspecionados contra o resto ===')
por_bairro = (
    fontes['cadastro']
    .select('id_uc', F.concat_ws(' / ', 'municipio', 'bairro').alias('bairro'))
    .join(fontes['inspecoes'].select('id_uc', 'resultado'), 'id_uc', 'left')
    .groupBy('bairro')
    .agg(
        F.countDistinct('id_uc').alias('ucs'),
        F.count('resultado').alias('inspecoes'),
        F.sum((F.col('resultado') == CONFIRMADA).cast('int')).alias('confirmadas'),
    )
    .withColumn('ordem', F.row_number().over(Window.orderBy(F.desc('inspecoes'), 'bairro')))
)
display(
    por_bairro.withColumn(
        'grupo', F.when(F.col('ordem') <= 4, F.col('bairro')).otherwise('demais bairros')
    )
    .groupBy('grupo')
    .agg(
        F.count(F.lit(1)).alias('bairros'),
        F.sum('ucs').alias('ucs'),
        F.sum('inspecoes').alias('inspecoes'),
        F.sum('confirmadas').alias('confirmadas'),
    )
    .orderBy(F.desc('inspecoes'))
)

print('=== 4) Situação das UCs com queda ≥ 40%: fila e exclusões ===')
display(
    candidatas.filter(F.col('score').isNotNull())
    .groupBy('situacao')
    .agg(
        F.count(F.lit(1)).alias('ucs'),
        F.round(F.sum('energia_kwh'), 2).alias('energia_kwh'),
        F.round(F.sum('energia_reais'), 2).alias('energia_reais'),
        F.round(F.avg('score'), 1).alias('score_medio'),
    )
    .orderBy('situacao')
)
fila = tabelas['fila_inspecao']
totais = fila.agg(
    F.count(F.lit(1)).alias('ucs'),
    F.sum('energia_recuperar_kwh').alias('kwh'),
    F.sum('energia_recuperar_reais').alias('reais'),
).first()
print(
    f'Fila: {totais["ucs"]:,} UCs | {totais["kwh"]:,} kWh | R$ {totais["reais"]:,} '
    f'(totais de controle do manifesto)'
)
print(
    'Origem da tarifa média das UCs da fila: '
    + ', '.join(
        f'{r["origem_tarifa"]}={r["n"]}'
        for r in candidatas.filter(F.col('situacao').startswith('FILA'))
        .groupBy('origem_tarifa')
        .agg(F.count(F.lit(1)).alias('n'))
        .orderBy('origem_tarifa')
        .collect()
    )
)

print('=== 5) As 10 primeiras UCs da fila ===')
display(fila.orderBy(F.desc('score'), 'id_uc').limit(10))

print('=== 6) Tarifa média das UCs da fila por subgrupo (R$/kWh, últimos 12 meses) ===')
display(
    candidatas.filter(F.col('situacao').startswith('FILA'))
    .join(fontes['cadastro'].select('id_uc', 'subgrupo_tarifario'), 'id_uc')
    .groupBy('subgrupo_tarifario')
    .agg(
        F.count(F.lit(1)).alias('ucs'),
        F.round(F.expr('percentile(tarifa, 0.5)'), 4).alias('tarifa_mediana'),
    )
    .orderBy('subgrupo_tarifario')
)

print('=== 7) Consumo médio (kWh/mês) antes e depois da inspeção, por resultado ===')
relativo = F.months_between('mes', F.trunc('data', 'MM'))
display(
    fontes['inspecoes']
    .select('id_uc', 'data', 'resultado')
    .join(mensal, 'id_uc')
    .withColumn(
        'periodo',
        F.when(relativo < -12, '1. 13 a 24 meses antes')
        .when(relativo < -6, '2. 7 a 12 meses antes')
        .when(relativo < -3, '3. 4 a 6 meses antes')
        .when(relativo < 0, '4. 1 a 3 meses antes')
        .when(relativo < 6, '5. até 5 meses depois')
        .otherwise('6. 6 meses ou mais depois'),
    )
    .filter(relativo >= -24)
    .groupBy('periodo')
    .agg(
        F.round(F.avg(F.when(F.col('resultado') == CONFIRMADA, F.col('kwh'))), 0).alias(
            'confirmadas'
        ),
        F.round(F.avg(F.when(F.col('resultado') == SEM_IRREGULARIDADE, F.col('kwh'))), 0).alias(
            'sem_irregularidade'
        ),
    )
    .orderBy('periodo')
)

print('=== 8) Medição antiga do acerto: P1 (12 meses antes da referência) contra P0, sem GD ===')
# A janela P1 inclui meses depois das inspeções (ago a dez/2025): é a comparação que deu
# "92% de acerto com queda > 10%" nos docs de overview, e não serve para a fila
por_periodo = mensal.groupBy('id_uc').agg(
    F.avg(
        F.when(
            (F.col('mes') >= F.add_months(F.lit(mes_ref), -MESES_ESTIMADAS))
            & (F.col('mes') < F.lit(mes_ref)),
            F.col('kwh'),
        )
    ).alias('p1'),
    F.avg(
        F.when(F.col('mes_referencia').between(INICIO_COMPARACAO, FIM_COMPARACAO), F.col('kwh'))
    ).alias('p0'),
)
display(
    fontes['inspecoes']
    .join(fontes['gd_uc'].select('id_uc'), 'id_uc', 'left_anti')
    .join(por_periodo.filter(F.col('p0') > 0), 'id_uc')
    .withColumn(
        'queda_p1_p0',
        F.when(1 - F.col('p1') / F.col('p0') > 0.10, 'mais de 10%').otherwise('até 10%'),
    )
    .groupBy('queda_p1_p0')
    .agg(
        F.count(F.lit(1)).alias('inspecoes'),
        F.sum((F.col('resultado') == CONFIRMADA).cast('int')).alias('confirmadas'),
        acerto.alias('pct_acerto'),
    )
    .orderBy('queda_p1_p0')
)

=== 1) Acerto das inspeções por queda de consumo antes da inspeção (contra o P0) ===


faixa_queda,inspecoes,confirmadas,pct_acerto,pct_eletromecanico,pct_outros_medidores,pct_estimadas_30_ou_mais,pct_estimadas_menos_30
0. UC com GD,31,1,3.2,0.0,6.3,0.0,5.6
0. sem leituras suficientes,11,6,54.5,80.0,33.3,80.0,0.0
1. menos de 30%,274,22,8.0,7.3,8.4,6.0,8.9
2. de 30% a 40%,123,9,7.3,2.3,10.1,5.6,8.0
3. de 40% a 50%,124,69,55.6,75.6,10.5,78.8,13.6
4. 50% ou mais,826,764,92.5,92.8,50.0,92.3,93.4


=== 2) Acerto das inspeções por faixa de score (UCs sem GD com queda ≥ 40%) ===


faixa_score,inspecoes,confirmadas,pct_acerto
1. 90 ou mais,657,607,92.4
2. 75 a 90,166,156,94.0
3. 60 a 75,82,64,78.0
4. menos de 60,45,6,13.3


=== 3) Concentração das inspeções: os 4 bairros mais inspecionados contra o resto ===


/databricks/python/lib/python3.11/site-packages/pyspark/sql/connect/expressions.py:1017: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


... (7 outputs left out)

=== 7) Consumo médio (kWh/mês) antes e depois da inspeção, por resultado ===


periodo,confirmadas,sem_irregularidade
1. 13 a 24 meses antes,491.0,381.0
2. 7 a 12 meses antes,549.0,411.0
3. 4 a 6 meses antes,367.0,340.0
4. 1 a 3 meses antes,190.0,297.0
5. até 5 meses depois,270.0,392.0
6. 6 meses ou mais depois,232.0,293.0


=== 8) Medição antiga do acerto: P1 (12 meses antes da referência) contra P0, sem GD ===


queda_p1_p0,inspecoes,confirmadas,pct_acerto
até 10%,448,36,8.0
mais de 10%,900,829,92.1


## 3. Publicação Delta com `RESTORE`

A publicação só ocorre quando `modo=publicar`. Antes da primeira escrita, registra a versão atual de cada tabela de destino. Depois de gravar, confere o schema e o conteúdo gravados e repete a validação da seção 1 sobre as tabelas persistidas; se a escrita ou a conferência falhar, restaura os destinos afetados com `RESTORE TABLE ... TO VERSION AS OF` (ou `DROP TABLE IF EXISTS` na primeira carga).

In [0]:
# ruff: noqa: F821
if config['modo'] == 'validar':
    print('Validação concluída. Nenhum schema ou tabela foi gravado.')
else:
    if any(dbutils.widgets.get(nome).strip() != valor for nome, valor in config.items()):
        raise RuntimeError(
            'Parâmetros mudaram durante a execução. Execute novamente desde o início.'
        )
    destinos = {nome: f'{schema_gold}.`{nome}`' for nome in tabelas}
    for chave, fonte in fontes_lidas.items():
        atual = spark.sql(f'DESCRIBE DETAIL {fonte["tabela"]}').first()
        if atual['id'] != fonte['id_delta']:
            raise RuntimeError(f'A identidade da tabela fonte mudou: {chave}')
    if set(destinos.values()) & {fonte['tabela'] for fonte in fontes_lidas.values()}:
        raise RuntimeError('Uma tabela de destino coincide com uma fonte')

    spark.sql(f'CREATE SCHEMA IF NOT EXISTS {schema_gold}')
    persistidas, versoes_gold = publicar_delta(
        spark,
        tabelas,
        destinos,
        lambda gravadas: validar_gold(gravadas, fontes, 'tabela gravada'),
    )
    for nome, info in versoes_gold.items():
        print(f'Publicada: {info["tabela"]} (versão {info["versao_delta"]})')

[OK   ] (tabela gravada) inspecoes_historicas: coluna obrigatória nula: 0
[OK   ] (tabela gravada) inspecoes_historicas: id_inspecao fora do padrão: 0
[OK   ] (tabela gravada) inspecoes_historicas: id_uc fora do padrão: 0
[OK   ] (tabela gravada) inspecoes_historicas: inspetor fora do padrão INSP-001 a INSP-040: 0
[OK   ] (tabela gravada) inspecoes_historicas: resultado fora do domínio: 0
[OK   ] (tabela gravada) inspecoes_historicas: kwh_recuperado negativo: 0
[OK   ] (tabela gravada) inspecoes_historicas: confirmada sem tipo de irregularidade: 0
[OK   ] (tabela gravada) inspecoes_historicas: sem irregularidade com tipo ou energia recuperada: 0
[OK   ] (tabela gravada) fila_inspecao: coluna nula: 0
[OK   ] (tabela gravada) fila_inspecao: id_uc fora do padrão: 0
[OK   ] (tabela gravada) fila_inspecao: score fora de 0 a 100: 0
[OK   ] (tabela gravada) fila_inspecao: motivos vazio ou com texto vazio: 0
[OK   ] (tabela gravada) fila_inspecao: energia a recuperar negativa: 0
[OK   ] (tabel